# Kiểm chứng lựa chọn Loss Function bằng Validation (chống rò rỉ)

**Dự án Tốt nghiệp - Energy Forecasting - Nhóm The Outliers**

## 1. Mục đích

Notebook này **KHÔNG train lại gì cả** — chỉ load lại 6 model `.pkl` đã train sẵn (MAE/Huber/MSE × H1/H4) và tính WAPE trên `v3_val_selected`, tập tách biệt với train và test, để xem model nào thắng thật.

Lý do cần làm: phát hiện lỗi rò rỉ dữ liệu — bước chọn loss function trước đây đang dùng nhầm tập test thay vì validation (`metrics_val.json` bị ghi từ metric tính trên `test_h`). Notebook này verify lại bằng validation thật, không đụng test, chạy trong vài giây.

## 2. Import và cấu hình đường dẫn

In [2]:
import os
import json
import pickle
import numpy as np
import pandas as pd

BASE = '/home/tandat/Desktop/Du_An_Tot_Nghiep_v3'
VAL_PATH = f'{BASE}/data/model/v3/05_selected/v3_val_selected.parquet'
TRAIN_DIR = f'{BASE}/data/model/v3/06_train'
EPS_ELEV = 0.05
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'

print('Da cau hinh duong dan. VAL_PATH =', VAL_PATH)
print('TRAIN_DIR =', TRAIN_DIR)

Da cau hinh duong dan. VAL_PATH = /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v3/05_selected/v3_val_selected.parquet
TRAIN_DIR = /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v3/06_train


## 3. Danh sách cột dịch theo horizon (`_mt`)

Sao chép đúng từ `04_1_train_mae.py` (COT_TAT_DINH) — các cột này được dịch tới thời điểm T+h thành cột mới hậu tố `_mt`, giống hệt hàm `them_muc_tieu()` trong pipeline training thật.

In [3]:
COT_TAT_DINH = [
    'solar_elevation', 'solar_azimuth', 'azimuth_sin', 'azimuth_cos', 'sin_elevation',
    'ghi_cs', 'clearsky_proxy', 'ky_vong', 'ty_le_bao_hoa',
    'minute_of_day', 'hour_of_day', 'hour_bucket_model', 'hour', 'hour_sin', 'hour_cos',
    'minute', 'day', 'day_of_week', 'month', 'day_of_year', 'doy_sin', 'doy_cos',
]
print(f'Co {len(COT_TAT_DINH)} cot se duoc dich thanh dac trung _mt.')

Co 22 cot se duoc dich thanh dac trung _mt.


## 4. Hàm chuẩn hoá mục tiêu và tính WAPE

Giống hệt `mau_chuan_hoa()` trong script training gốc.

In [4]:
def mau_chuan_hoa(df):
    """Mau so de chuan hoa muc tieu: quy mo tram nhan sin(goc cao mat troi)."""
    return (df['site_scale'] * np.clip(df['sin_elevation'], EPS_ELEV, None)).to_numpy()


def compute_wape(yt, yp):
    yt = np.asarray(yt, dtype=float)
    yp = np.asarray(yp, dtype=float)
    denom = np.sum(np.abs(yt))
    return float(np.sum(np.abs(yt - yp)) / denom * 100.0) if denom > 0 else float('nan')

print('Da dinh nghia mau_chuan_hoa va compute_wape.')

Da dinh nghia mau_chuan_hoa va compute_wape.


## 5. Nạp tập validation holdout

Đọc trực tiếp `v3_val_selected.parquet`, tự tạo `y_true` (dịch target theo horizon, giống `them_muc_tieu()`) và các cột `_mt` cần thiết. Tập này không dùng để train hoặc test.

In [5]:
def load_val_holdout(features, horizon_steps):
    # Can doc them cac cot GOC (khong _mt) cua COT_TAT_DINH de dich thanh _mt.
    base_needed = [c[:-3] if c.endswith('_mt') and c[:-3] in COT_TAT_DINH else c for c in features]
    need = list(dict.fromkeys(
        base_needed + [SITE_COL, TIMESTAMP_COL, TARGET_COL, 'site_scale', 'sin_elevation',
                       'tran_cong_suat', 'energy_source', 'is_daylight']))
    d = pd.read_parquet(VAL_PATH)
    need = [c for c in need if c in d.columns]
    d = d[need].sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)
    h = int(horizon_steps)
    d['y_true'] = d.groupby(SITE_COL)[TARGET_COL].shift(-h)
    g = d.groupby(SITE_COL)
    for c in COT_TAT_DINH:
        if c in d.columns and f'{c}_mt' in features:
            d[f'{c}_mt'] = g[c].shift(-h)
    d = d.dropna(subset=['y_true'])
    return d[(d['site_scale'] > 0) & (d['sin_elevation'] > EPS_ELEV)].copy()

print('Da dinh nghia load_val_holdout (v3_val_selected).')

Da dinh nghia load_val_holdout (v3_val_selected).


## 6. Đánh giá 1 model (load lại `.pkl`, dự báo trên validation, tính metric)

In [6]:
_VAL_CACHE = {}


def _metric_1_scope(yt, yp):
    yt = np.asarray(yt, dtype=float); yp = np.asarray(yp, dtype=float)
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2))) if len(yt) else float('nan')
    mae = float(np.mean(np.abs(yt - yp))) if len(yt) else float('nan')
    ss_res = np.sum((yt - yp) ** 2); ss_tot = np.sum((yt - np.mean(yt)) ** 2)
    r2 = float(1 - ss_res / ss_tot) if len(yt) and ss_tot > 0 else float('nan')
    return {'wape': compute_wape(yt, yp), 'rmse': rmse, 'mae': mae, 'r2': r2, 'n': int(len(yt))}


def eval_one(loss_name, h_label):
    cfg_path = f'{TRAIN_DIR}/{loss_name}/{h_label}/model_config.json'
    pkl_path = f'{TRAIN_DIR}/{loss_name}/{h_label}/model.pkl'
    cfg = json.load(open(cfg_path))
    features = cfg['features']
    medians = cfg['feature_medians']
    horizon_steps = cfg['horizon_steps']

    with open(pkl_path, 'rb') as f:
        model = pickle.load(f)

    cache_key = int(horizon_steps)
    if cache_key not in _VAL_CACHE:
        _VAL_CACHE[cache_key] = load_val_holdout(features, horizon_steps)
    val = _VAL_CACHE[cache_key]

    missing_feats = [c for c in features if c not in val.columns]
    if missing_feats:
        raise KeyError(f'Thieu {len(missing_feats)} dac trung: {missing_feats[:5]}...')

    X = val[features].fillna(pd.Series(medians)).astype(np.float32)
    k_pred = np.clip(model.predict(X), 0, 1.5)
    y_pred = np.minimum(k_pred * mau_chuan_hoa(val), val['tran_cong_suat'].to_numpy() * 1.02)
    y_pred = np.where(val['sin_elevation'].to_numpy() <= EPS_ELEV, 0.0, y_pred)
    y_true = val['y_true'].to_numpy()

    scope_all = _metric_1_scope(y_true, y_pred)

    mask = np.ones(len(val), dtype=bool)
    if 'energy_source' in val.columns:
        mask &= (val['energy_source'] == 'measured').to_numpy()
    if 'is_daylight' in val.columns:
        mask &= val['is_daylight'].fillna(False).astype(bool).to_numpy()
    scope_md = _metric_1_scope(y_true[mask], y_pred[mask])

    # Ghi metric tren holdout v3_val_selected de notebook 07 doc dung o lan chay tiep theo.
    out_dir = f'{TRAIN_DIR}/{loss_name}/{h_label}'
    with open(f'{out_dir}/metrics_val.json', 'w', encoding='utf-8') as f:
        json.dump({'horizon_steps': int(horizon_steps), 'loss_name': loss_name,
                   'feature_set_name': '', 'measured_daylight': scope_md,
                   'all': scope_all}, f, indent=2, ensure_ascii=False, default=str)

    return {'loss': loss_name, 'horizon': h_label, 'n_rows_val_total': len(val),
            'n_measured_daylight': scope_md['n'], 'wape_val_%': round(scope_md['wape'], 4),
            'rmse_val': round(scope_md['rmse'], 4), 'mae_val': round(scope_md['mae'], 4),
            'r2_val': round(scope_md['r2'], 4)}

print('Da dinh nghia eval_one (v3_val_selected -> metrics_val.json).')

Da dinh nghia eval_one (v3_val_selected -> metrics_val.json).


**Lưu ý:** `eval_one()` ghi `metrics_val.json` sau khi chấm trên `v3_val_selected`, nên notebook 07 đọc đúng holdout validation.

## 7. Chạy đánh giá cho cả 6 model và xác định model thắng thật

In [7]:
rows = []
for loss in ['mae', 'huber', 'mse']:
    for h in ['h1', 'h4']:
        try:
            r = eval_one(loss, h)
            rows.append(r)
            print(f"{loss:6s} {h}: WAPE_val={r['wape_val_%']:>8.4f}%  RMSE_val={r['rmse_val']:>8.4f}  "
                  f"MAE_val={r['mae_val']:>8.4f}  R2_val={r['r2_val']:>7.4f}  "
                  f"(n={r['n_measured_daylight']:,} / {r['n_rows_val_total']:,})")
        except Exception as e:
            print(f'{loss:6s} {h}: LOI - {e}')

df_ket_qua = pd.DataFrame(rows)
display(df_ket_qua)

mae    h1: WAPE_val= 21.0302%  RMSE_val=  3.9567  MAE_val=  1.6317  R2_val= 0.8996  (n=220,807 / 245,147)
mae    h4: WAPE_val= 25.2243%  RMSE_val=  4.4527  MAE_val=  1.9382  R2_val= 0.8743  (n=220,687 / 245,021)
huber  h1: WAPE_val= 21.4990%  RMSE_val=  3.8844  MAE_val=  1.6681  R2_val= 0.9032  (n=220,807 / 245,147)
huber  h4: WAPE_val= 25.7314%  RMSE_val=  4.4211  MAE_val=  1.9771  R2_val= 0.8761  (n=220,687 / 245,021)
mse    h1: WAPE_val= 21.6301%  RMSE_val=  3.9099  MAE_val=  1.6783  R2_val= 0.9019  (n=220,807 / 245,147)
mse    h4: WAPE_val= 25.6407%  RMSE_val=  4.4194  MAE_val=  1.9702  R2_val= 0.8762  (n=220,687 / 245,021)


,loss,horizon,n_rows_val_total,n_measured_daylight,wape_val_%,rmse_val,mae_val,r2_val
0,mae,h1,245147,220807,21.0302,3.9567,1.6317,0.8996
1,mae,h4,245021,220687,25.2243,4.4527,1.9382,0.8743
2,huber,h1,245147,220807,21.4990,3.8844,1.6681,0.9032
3,huber,h4,245021,220687,25.7314,4.4211,1.9771,0.8761
4,mse,h1,245147,220807,21.6301,3.9099,1.6783,0.9019
5,mse,h4,245021,220687,25.6407,4.4194,1.9702,0.8762


In [8]:
print('=== MODEL THANG THAT TREN VALIDATION (WAPE thap nhat, khong dung tap test) ===')
nguoi_thang = {}
for h in ['h1', 'h4']:
    sub = [r for r in rows if r['horizon'] == h]
    if sub:
        best = min(sub, key=lambda r: r['wape_val_%'])
        nguoi_thang[h] = best['loss']
        print(f"{h}: {best['loss'].upper()} thang voi WAPE_val = {best['wape_val_%']:.4f}%")
print()
print('Ket qua nay dung de CHON model, khong dung de bao cao headline.')
print('So lieu headline chinh thuc van lay tu ket_qua.json / metrics_overall.json tren tap TEST')
print('cua dung model vua thang o day (chi doc 1 lan, khong dung de chon lai).')

=== MODEL THANG THAT TREN VALIDATION (WAPE thap nhat, khong dung tap test) ===
h1: MAE thang voi WAPE_val = 21.0302%
h4: MAE thang voi WAPE_val = 25.2243%

Ket qua nay dung de CHON model, khong dung de bao cao headline.
So lieu headline chinh thuc van lay tu ket_qua.json / metrics_overall.json tren tap TEST
cua dung model vua thang o day (chi doc 1 lan, khong dung de chon lai).


## 8. Xuất kết quả ra file (để trích dẫn trong report)

In [9]:
OUT_PATH = f'{BASE}/data/model/v3/07_final_test/val_model_selection_check.json'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump({'ket_qua_tung_model': rows, 'model_thang': nguoi_thang}, f, indent=2, ensure_ascii=False)
print(f'Da luu: {OUT_PATH}')

Da luu: /home/tandat/Desktop/Du_An_Tot_Nghiep_v3/data/model/v3/07_final_test/val_model_selection_check.json
